In [1]:
import pandas as pd
# Load data into pandas DataFrame from "/lakehouse/default/Files/acto_obras/silver/silver_acto_gesta_obras_santos_etapas.parquet"
df_obras_etapas = pd.read_parquet("/lakehouse/default/Files/acto_obras/silver/silver_acto_gesta_obras_santos_etapas.parquet")
df_obras_solicitacoes = pd.read_parquet("/lakehouse/default/Files/acto_obras/silver/silver_acto_gesta_obras_santos_solicitacoes.parquet")

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 3, Finished, Available, Finished)

In [2]:
%run ./nb_utils_api_acto_gestao

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 7, Finished, Available, Finished)

In [3]:
TOKEN = TOKEN_SANTOS_OBRAS

PARAM_LOGIN = "3997"
ORIGIN_URL = "https://gestaoaprovasantos.acto.net.br"
APP_ID = "86bf9fc6-78ad-4a65-89e8-8c91f8eac43d"  # App_Id do token JWT
BASE_API_URL = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net"

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Authorization": f"Bearer {TOKEN}",
    "App_Id": APP_ID,
    "ApplicationId": APP_ID,
    "Origin": ORIGIN_URL,
    "Referer": f"{ORIGIN_URL}/",
    "PARAM_LOGIN": PARAM_LOGIN,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Content-Type": "application/json",
}


df_tempo_etapa = obter_dados_etapa_atual(TOKEN, [
    4803, 4804, 5605, 5625, 5626, 
    5627, 5628, 5677, 5679, 5685, 
    5686, 5693, 5725, 5755, 5964, 
    6093, 6113, 6326, 6383, 6513, 
    6738, 6783, 6963, 7523, 8134, 12804, 7243
])
for col in df_tempo_etapa.filter(like="data").columns.tolist():
    df_tempo_etapa[col] = pd.to_datetime(df_tempo_etapa[col], format="ISO8601")

df_tempo_etapa = df_tempo_etapa.drop(columns=["notifications", "isValid"])

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 8, Finished, Available, Finished)

# ==========================================
# MAPEAMENTO E PREPARAÇÃO DOS DADOS
# ==========================================

In [4]:
df_tempo_etapa.columns

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 9, Finished, Available, Finished)

Index(['codEtapa', 'etapa', 'servico', 'seqFluxo', 'dataCriacaoOS',
       'dataFinalizacaoOS', 'dataEtapaInicio', 'dataEtapaFim',
       'dataAtenderEtapa', 'tempoExecucao', 'tempoExecucaoHoras', 'status',
       'executor'],
      dtype='object')

In [5]:
# ==========================================
# MAPEAMENTO DE COLUNAS
# ==========================================
# Criar DataFrame processado com mapeamento de colunas da API

print("🔧 Mapeando colunas da API para o formato esperado...")

# Criar DataFrame processado com mapeamento de colunas
df = pd.DataFrame()

# Mapeamento de colunas da API para o formato esperado
df["os"] = df_tempo_etapa["seqFluxo"].astype(str).str.strip()
df["etapa"] = df_tempo_etapa["etapa"].astype(str).str.strip()
df["servico"] = df_tempo_etapa["servico"].astype(str).str.strip()
df["data_criacao_os"] = df_tempo_etapa["dataCriacaoOS"]
df["data_inicio_etapa"] = df_tempo_etapa["dataEtapaInicio"]
df["data_atendimento_etapa"] = df_tempo_etapa["dataAtenderEtapa"]
df["data_fim_etapa"] = df_tempo_etapa["dataEtapaFim"]
df["data_finalizacao_os"] = df_tempo_etapa["dataFinalizacaoOS"]
df["tempo_execucao"] = df_tempo_etapa["tempoExecucao"].astype(str).str.strip()
df["status"] = df_tempo_etapa["status"].astype(str).str.strip()
df["executor"] = df_tempo_etapa["executor"].astype(str).str.strip()

# Garantir que as colunas de data estão como datetime (necessário para cálculos)
colunas_data = [
    "data_criacao_os",
    "data_inicio_etapa",
    "data_atendimento_etapa",
    "data_fim_etapa",
    "data_finalizacao_os"
]

for c in colunas_data:
    if c in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = pd.to_datetime(df[c], errors='coerce')

print(f"✅ Mapeamento concluído! Total de registros: {len(df):,}")
print()



StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 10, Finished, Available, Finished)

🔧 Mapeando colunas da API para o formato esperado...
✅ Mapeamento concluído! Total de registros: 71,500



In [6]:
# print(f"✅ Dados mapeados: {len(df):,} registros, {len(df.columns)} colunas")
# print()
# print("👀 Prévia dos dados mapeados:")
# display(df.head())

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 11, Finished, Available, Finished)

In [7]:
# ==========================================
# CÁLCULO DE DURAÇÃO (ANTES DE CONVERTER DATAS PARA STRING)
# ==========================================
# IMPORTANTE: Este cálculo deve ser feito enquanto as datas ainda são datetime

print("⏱️ Calculando duração entre datas...")

# Cálculo da duração entre datas: fim - início
df["duracao_timedelta"] = df["data_fim_etapa"] - df["data_inicio_etapa"]

# Converte o timedelta para dias com casas decimais
df["duracao_dias_preciso"] = df["duracao_timedelta"].dt.total_seconds() / 86400

# Arredondamento tradicional
df["duracao_dias_int"] = df["duracao_dias_preciso"].round().astype("Int64")

# Remove coluna não suportada pelo Delta Lake
if "duracao_timedelta" in df.columns:
    df = df.drop(columns=["duracao_timedelta"])

print("✅ Cálculo de duração concluído!")
print()

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 12, Finished, Available, Finished)

⏱️ Calculando duração entre datas...
✅ Cálculo de duração concluído!



In [8]:
# ==========================================
# TRATAMENTO DE DATAS (CONVERSÃO PARA STRING)
# ==========================================
# IMPORTANTE: Converter para string APÓS todos os cálculos com datetime

colunas_data = [
    "data_criacao_os",
    "data_inicio_etapa",
    "data_atendimento_etapa",
    "data_fim_etapa",
    "data_finalizacao_os"
]

print("📅 Convertendo colunas de data para string ISO...")

for c in colunas_data:
    if c in df.columns:
        # Garantir que está como datetime antes de formatar
        if not pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = pd.to_datetime(df[c], errors='coerce')
        
        # Formata como string ISO (YYYY-MM-DD HH:MM:SS) para garantir interpretação correta
        # O formato ISO é universal e não depende de localização
        df[c] = df[c].dt.strftime('%Y-%m-%d %H:%M:%S')
        
print("✅ Conversão de datas concluída!")
print()


StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 13, Finished, Available, Finished)

📅 Convertendo colunas de data para string ISO...
✅ Conversão de datas concluída!



# ==========================================
# MERGE COM TABELA AUXILIAR
# ==========================================


In [9]:
# ==========================================
# MERGE COM TABELA AUXILIAR
# ==========================================

aux_path = "/lakehouse/default/Files/acto/PMS_AuxiliarPDR.xlsx"

aux_pdr = pd.read_excel(aux_path)

# Renomear colunas do auxiliar para padronizar
aux_pdr = aux_pdr.rename(columns={
    "Etapa": "etapa_pad",
    "AuxSetorResponsável": "aux_setor_responsavel",
    "AuxPDR": "aux_pdr"
})

# Padronizar etapas (uppercase e strip)
aux_pdr["etapa_pad"] = aux_pdr["etapa_pad"].astype(str).str.upper().str.strip()
df["etapa_pad"] = df["etapa"].astype(str).str.upper().str.strip()

# Merge
df = df.merge(
    aux_pdr,
    how="left",
    on="etapa_pad"
)

print("Merge com tabela auxiliar concluído!")
print(df.columns.tolist())

# Remover colunas auxiliares e desnecessárias
df = df.drop(columns=["etapa_pad", "Zona"])

print("Colunas ajustadas para criação da tabela GOLD!")


StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 14, Finished, Available, Finished)

Merge com tabela auxiliar concluído!
['os', 'etapa', 'servico', 'data_criacao_os', 'data_inicio_etapa', 'data_atendimento_etapa', 'data_fim_etapa', 'data_finalizacao_os', 'tempo_execucao', 'status', 'executor', 'duracao_dias_preciso', 'duracao_dias_int', 'etapa_pad', 'aux_setor_responsavel', 'aux_pdr', 'Zona']
Colunas ajustadas para criação da tabela GOLD!


In [10]:
df["aux_setor_responsavel"].count()

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 15, Finished, Available, Finished)

67477

In [11]:
# ==========================================
# VERIFICAÇÃO DOS VALORES ÚNICOS DO MERGE
# ==========================================
# Verificar todos os valores únicos das colunas do merge e seus matches

print("🔍 Verificando valores únicos das colunas do merge...")
print()

# Verificar aux_setor_responsavel
if "aux_setor_responsavel" in df.columns:
    print("=" * 80)
    print("📊 COLUNA: aux_setor_responsavel")
    print("=" * 80)
    
    valores_unicos = df["aux_setor_responsavel"].value_counts(dropna=False).sort_index()
    total_registros = len(df)
    total_com_valor = df["aux_setor_responsavel"].notna().sum()
    total_sem_valor = df["aux_setor_responsavel"].isna().sum()
    
    print(f"Total de registros: {total_registros:,}")
    print(f"Registros COM valor: {total_com_valor:,} ({total_com_valor/total_registros*100:.2f}%)")
    print(f"Registros SEM valor (NULL): {total_sem_valor:,} ({total_sem_valor/total_registros*100:.2f}%)")
    print()
    print("Valores únicos e suas contagens:")
    print("-" * 80)
    for valor, count in valores_unicos.items():
        pct = (count / total_registros) * 100
        if pd.isna(valor):
            print(f"  NULL: {count:,} registros ({pct:.2f}%)")
        else:
            print(f"  '{valor}': {count:,} registros ({pct:.2f}%)")
    print()
else:
    print("⚠️  Coluna 'aux_setor_responsavel' não encontrada!")
    print()

# Verificar aux_pdr
if "aux_pdr" in df.columns:
    print("=" * 80)
    print("📊 COLUNA: aux_pdr")
    print("=" * 80)
    
    valores_unicos = df["aux_pdr"].value_counts(dropna=False).sort_index()
    total_registros = len(df)
    total_com_valor = df["aux_pdr"].notna().sum()
    total_sem_valor = df["aux_pdr"].isna().sum()
    
    print(f"Total de registros: {total_registros:,}")
    print(f"Registros COM valor: {total_com_valor:,} ({total_com_valor/total_registros*100:.2f}%)")
    print(f"Registros SEM valor (NULL): {total_sem_valor:,} ({total_sem_valor/total_registros*100:.2f}%)")
    print()
    print("Valores únicos e suas contagens:")
    print("-" * 80)
    for valor, count in valores_unicos.items():
        pct = (count / total_registros) * 100
        if pd.isna(valor):
            print(f"  NULL: {count:,} registros ({pct:.2f}%)")
        else:
            print(f"  '{valor}': {count:,} registros ({pct:.2f}%)")
    print()
else:
    print("⚠️  Coluna 'aux_pdr' não encontrada!")
    print()

# Verificar combinação das duas colunas
if "aux_setor_responsavel" in df.columns and "aux_pdr" in df.columns:
    print("=" * 80)
    print("📊 COMBINAÇÃO: aux_setor_responsavel + aux_pdr")
    print("=" * 80)
    
    combinacoes = df.groupby(["aux_setor_responsavel", "aux_pdr"], dropna=False).size().reset_index(name="count")
    combinacoes = combinacoes.sort_values("count", ascending=False)
    
    print(f"Total de combinações únicas: {len(combinacoes):,}")
    print()
    print("Top 20 combinações mais frequentes:")
    print("-" * 80)
    for idx, row in combinacoes.head(20).iterrows():
        setor = row["aux_setor_responsavel"] if pd.notna(row["aux_setor_responsavel"]) else "NULL"
        pdr = row["aux_pdr"] if pd.notna(row["aux_pdr"]) else "NULL"
        count = row["count"]
        pct = (count / len(df)) * 100
        print(f"  Setor: '{setor}' | PDR: '{pdr}' → {count:,} registros ({pct:.2f}%)")
    print()

print("✅ Verificação concluída!")
print()

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 16, Finished, Available, Finished)

🔍 Verificando valores únicos das colunas do merge...

📊 COLUNA: aux_setor_responsavel
Total de registros: 71,500
Registros COM valor: 67,477 (94.37%)
Registros SEM valor (NULL): 4,023 (5.63%)

Valores únicos e suas contagens:
--------------------------------------------------------------------------------
  'COAP': 3 registros (0.00%)
  'CONDEPASA': 3 registros (0.00%)
  'Chefia SEFISO': 1,227 registros (1.72%)
  'DECONTE': 69 registros (0.10%)
  'Pareceres': 1,117 registros (1.56%)
  'SAAF': 4,893 registros (6.84%)
  'SEAP-CB': 562 registros (0.79%)
  'SEAP-CC': 372 registros (0.52%)
  'SEAP-SL': 995 registros (1.39%)
  'SEAP-TG': 549 registros (0.77%)
  'SECATEM': 247 registros (0.35%)
  'SEDURB': 10 registros (0.01%)
  'SEFISO': 3,780 registros (5.29%)
  'SEINST': 70 registros (0.10%)
  'SEINST-Chefia': 206 registros (0.29%)
  'SEOBE': 9 registros (0.01%)
  'SEONT': 8,550 registros (11.96%)
  'SEONT CHEFIA': 3 registros (0.00%)
  'SEONT-Chefia': 3,787 registros (5.30%)
  'SEONT-Chef

In [12]:
# Verificar DataFrame antes de salvar
print(f"Registros: {len(df):,}")
print(f"Colunas: {len(df.columns)}")
print(f"\nColunas: {list(df.columns)}")
print(f"\nPrimeiras 5 linhas:")
display(df.head())
print(f"\nTipos de dados:")
print(df.dtypes)


StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 17, Finished, Available, Finished)

Registros: 71,500
Colunas: 15

Colunas: ['os', 'etapa', 'servico', 'data_criacao_os', 'data_inicio_etapa', 'data_atendimento_etapa', 'data_fim_etapa', 'data_finalizacao_os', 'tempo_execucao', 'status', 'executor', 'duracao_dias_preciso', 'duracao_dias_int', 'aux_setor_responsavel', 'aux_pdr']

Primeiras 5 linhas:


SynapseWidget(Synapse.DataFrame, 264bd449-012f-4c2e-85f7-51a419ac3510)


Tipos de dados:
os                         object
etapa                      object
servico                    object
data_criacao_os            object
data_inicio_etapa          object
data_atendimento_etapa     object
data_fim_etapa             object
data_finalizacao_os        object
tempo_execucao             object
status                     object
executor                   object
duracao_dias_preciso      float64
duracao_dias_int            Int64
aux_setor_responsavel      object
aux_pdr                    object
dtype: object


In [13]:
# Cria o DataFrame Spark a partir do pandas
df_spark = spark.createDataFrame(df)

# Importar funções do Spark para manipulação de datas
from pyspark.sql import functions as F

# Mostra o schema para conferência
df_spark.printSchema()

# Permite evolução de schema e sobrescrita completa
(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_obras_tempo_etapa")
)

print("Tabela Gold criada com sucesso!")


StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 18, Finished, Available, Finished)

root
 |-- os: string (nullable = true)
 |-- etapa: string (nullable = true)
 |-- servico: string (nullable = true)
 |-- data_criacao_os: string (nullable = true)
 |-- data_inicio_etapa: string (nullable = true)
 |-- data_atendimento_etapa: string (nullable = true)
 |-- data_fim_etapa: string (nullable = true)
 |-- data_finalizacao_os: string (nullable = true)
 |-- tempo_execucao: string (nullable = true)
 |-- status: string (nullable = true)
 |-- executor: string (nullable = true)
 |-- duracao_dias_preciso: double (nullable = true)
 |-- duracao_dias_int: long (nullable = true)
 |-- aux_setor_responsavel: string (nullable = true)
 |-- aux_pdr: string (nullable = true)

Tabela Gold criada com sucesso!


In [14]:
%%sql
SELECT * FROM gold_obras_tempo_etapa

StatementMeta(, b5aabc4a-ff87-4773-9fc0-e89227466477, 19, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 15 fields>